# Medical Transcription — Exploratory Data Analysis

This notebook provides a comprehensive EDA of the medical transcription dataset.  
Dataset columns: **`label`** (1–4), **`description`**, **`text`** (transcription)  
Classes (from `classes.txt`): 1 = Surgery, 2 = Medical Records, 3 = Internal Medicine, 4 = Other

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# ── Consistent style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='tab10')
PALETTE = sns.color_palette('tab10', 4)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = Path('.')
TRAIN_CSV   = BASE / 'train.csv'
TEST_CSV    = BASE / 'test.csv'
CLASSES_TXT = BASE / 'classes.txt'
STOPWORDS_TXT = BASE / 'clinical-stopwords.txt'
VOCAB_TXT   = BASE / 'vocab.txt'

# ── Class mapping ─────────────────────────────────────────────────────────────
CLASS_NAMES = {}
with open(CLASSES_TXT) as f:
    for idx, line in enumerate(f, start=1):
        CLASS_NAMES[idx] = line.strip()
print('Classes:', CLASS_NAMES)

# ── Clinical stopwords ────────────────────────────────────────────────────────
STOPWORDS = set()
with open(STOPWORDS_TXT) as f:
    for line in f:
        w = line.strip()
        if w and not w.startswith('#'):
            STOPWORDS.add(w.lower())
print(f'Loaded {len(STOPWORDS):,} clinical stop-words')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Map numeric labels to class names
train_df['class'] = train_df['label'].map(CLASS_NAMES)
test_df['class']  = test_df['label'].map(CLASS_NAMES)

# Add character-count and word-count columns
train_df['char_len'] = train_df['text'].str.len()
train_df['word_len'] = train_df['text'].str.split().str.len()
test_df['char_len']  = test_df['text'].str.len()
test_df['word_len']  = test_df['text'].str.split().str.len()

print('Train:', train_df.shape, '  Test:', test_df.shape)
train_df.head(2)

---
## 1. Class Distribution

In [ ]:
counts = train_df['class'].value_counts().reindex(CLASS_NAMES.values())
pcts   = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=PALETTE)
axes[0].set_title('Class Distribution (Train Set)')
axes[0].set_xlabel('Medical Specialty')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=15)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 10,
                 f'{cnt}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution — Pie Chart (Train Set)')

plt.tight_layout()
plt.show()

---
## 2. Text Length Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram – character count
axes[0, 0].hist(train_df['char_len'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Character Count Distribution')
axes[0, 0].set_xlabel('Characters per Sample')
axes[0, 0].set_ylabel('Frequency')

# Histogram – word count
axes[0, 1].hist(train_df['word_len'], bins=50, color='darkorange', edgecolor='white')
axes[0, 1].set_title('Word Count Distribution')
axes[0, 1].set_xlabel('Words per Sample')
axes[0, 1].set_ylabel('Frequency')

# Box plot – word count per class
order = list(CLASS_NAMES.values())
sns.boxplot(data=train_df, x='class', y='word_len', order=order,
            palette=PALETTE, ax=axes[1, 0])
axes[1, 0].set_title('Word Count by Class (Box Plot)')
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Word Count')
axes[1, 0].tick_params(axis='x', rotation=15)

# Violin plot – word count per class
sns.violinplot(data=train_df, x='class', y='word_len', order=order,
               palette=PALETTE, ax=axes[1, 1], inner='quartile')
axes[1, 1].set_title('Word Count by Class (Violin Plot)')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Word Count')
axes[1, 1].tick_params(axis='x', rotation=15)

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 3. Word Frequency Analysis

Clinical stopwords are used to filter common, non-informative terms.

In [ ]:
def tokenize(text, stopwords=STOPWORDS):
    """Lowercase, keep only alphabetic tokens, remove stopwords."""
    tokens = re.findall(r'[a-z]+', str(text).lower())
    return [t for t in tokens if t not in stopwords and len(t) > 2]

TOP_N = 20

# ── Overall frequency ─────────────────────────────────────────────────────────
all_tokens = [t for text in train_df['text'] for t in tokenize(text)]
overall_freq = collections.Counter(all_tokens).most_common(TOP_N)
words, freqs = zip(*overall_freq)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(words[::-1], freqs[::-1], color='steelblue')
ax.set_title(f'Top {TOP_N} Most Frequent Words (Overall)')
ax.set_xlabel('Frequency')
ax.set_ylabel('Word')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-class frequency ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    cls_texts = train_df[train_df['label'] == label]['text']
    tokens = [t for text in cls_texts for t in tokenize(text)]
    top = collections.Counter(tokens).most_common(15)
    if top:
        ws, fs = zip(*top)
        ax.barh(ws[::-1], fs[::-1], color=PALETTE[label - 1])
    ax.set_title(f'Top 15 Words — {cls_name}')
    ax.set_xlabel('Frequency')

plt.suptitle('Top 15 Most Frequent Words per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Word Clouds

In [ ]:
wc_kwargs = dict(background_color='white', max_words=200,
                 stopwords=STOPWORDS, width=800, height=400,
                 colormap='viridis')

# Overall word cloud
all_text = ' '.join(train_df['text'].fillna(''))
wc = WordCloud(**wc_kwargs).generate(all_text)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Overall Word Cloud — Medical Transcriptions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class word clouds (2×2 grid)
colormaps = ['Blues', 'Oranges', 'Greens', 'Purples']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (label, cls_name), cmap in zip(axes.flat, CLASS_NAMES.items(), colormaps):
    cls_text = ' '.join(train_df[train_df['label'] == label]['text'].fillna(''))
    wc = WordCloud(**{**wc_kwargs, 'colormap': cmap}).generate(cls_text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(cls_name, fontsize=12)

plt.suptitle('Per-Class Word Clouds', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. N-gram Analysis

In [ ]:
def get_ngrams(texts, n=2, top_k=15, stopwords=STOPWORDS):
    """Return the top-k n-grams from a collection of texts."""
    ngrams = []
    for text in texts:
        tokens = tokenize(text, stopwords)
        ngrams.extend(zip(*[tokens[i:] for i in range(n)]))
    return collections.Counter(ngrams).most_common(top_k)

def plot_ngrams(ngram_counts, title, ax, color):
    if not ngram_counts:
        return
    labels = [' '.join(g) for g, _ in ngram_counts]
    vals   = [c for _, c in ngram_counts]
    ax.barh(labels[::-1], vals[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Count')

# Overall bigrams & trigrams
bigrams  = get_ngrams(train_df['text'], n=2)
trigrams = get_ngrams(train_df['text'], n=3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_ngrams(bigrams,  'Top 15 Bigrams (Overall)',  axes[0], 'steelblue')
plot_ngrams(trigrams, 'Top 15 Trigrams (Overall)', axes[1], 'darkorange')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class bigrams
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    cls_texts = train_df[train_df['label'] == label]['text']
    bg = get_ngrams(cls_texts, n=2, top_k=10)
    plot_ngrams(bg, f'Top 10 Bigrams — {cls_name}', ax, PALETTE[label - 1])
plt.suptitle('Top Bigrams per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Vocabulary Coverage

In [ ]:
vocab_data = []
for label, cls_name in CLASS_NAMES.items():
    cls_texts = train_df[train_df['label'] == label]['text']
    tokens = [t for text in cls_texts for t in tokenize(text)]
    vocab_data.append({
        'Class': cls_name,
        'Total Tokens': len(tokens),
        'Unique Tokens': len(set(tokens)),
        'Type-Token Ratio': round(len(set(tokens)) / max(len(tokens), 1), 4)
    })

vocab_df = pd.DataFrame(vocab_data)
print(vocab_df.to_string(index=False))

x = np.arange(len(vocab_df))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, vocab_df['Total Tokens'],  width, label='Total Tokens',  color='steelblue')
bars2 = ax.bar(x + width/2, vocab_df['Unique Tokens'], width, label='Unique Tokens', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(vocab_df['Class'], rotation=10)
ax.set_title('Vocabulary Coverage per Class')
ax.set_ylabel('Token Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Token Length Distribution & CDF

In [ ]:
token_counts = train_df['text'].apply(lambda t: len(tokenize(t)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(token_counts, bins=50, color='teal', edgecolor='white')
axes[0].set_title('Token Count Distribution per Sample')
axes[0].set_xlabel('Number of Tokens (after stopword removal)')
axes[0].set_ylabel('Frequency')

# CDF
sorted_tc = np.sort(token_counts)
cdf = np.arange(1, len(sorted_tc) + 1) / len(sorted_tc)
axes[1].plot(sorted_tc, cdf, color='teal', linewidth=2)
axes[1].axhline(0.90, color='red',    linestyle='--', label='90th percentile')
axes[1].axhline(0.95, color='orange', linestyle='--', label='95th percentile')
axes[1].set_title('CDF of Token Length')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Median tokens: {token_counts.median():.0f}")
print(f"90th percentile: {np.percentile(token_counts, 90):.0f}")
print(f"95th percentile: {np.percentile(token_counts, 95):.0f}")

---
## 8. TF-IDF Feature Correlation Heatmap

We compute TF-IDF for the top 30 terms and display a correlation matrix of those features.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=30,
    stop_words=list(STOPWORDS),
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
)
tfidf_matrix = vectorizer.fit_transform(train_df['text'].fillna(''))
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

corr = tfidf_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5,
            annot=False, fmt='.2f', cbar_kws={'shrink': 0.8})
ax.set_title('TF-IDF Feature Correlation Heatmap (Top 30 Terms)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 9. Train / Test Split Analysis

In [ ]:
train_counts = (train_df['class']
                .value_counts()
                .reindex(CLASS_NAMES.values(), fill_value=0))
test_counts  = (test_df['class']
                .value_counts()
                .reindex(CLASS_NAMES.values(), fill_value=0))

train_pct = train_counts / train_counts.sum() * 100
test_pct  = test_counts  / test_counts.sum()  * 100

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
axes[0].bar(x - width/2, train_counts.values, width, label='Train', color='steelblue')
axes[0].bar(x + width/2, test_counts.values,  width, label='Test',  color='darkorange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[0].set_title('Class Count — Train vs. Test')
axes[0].set_ylabel('Samples')
axes[0].legend()

# Percentage
axes[1].bar(x - width/2, train_pct.values, width, label='Train', color='steelblue')
axes[1].bar(x + width/2, test_pct.values,  width, label='Test',  color='darkorange')
axes[1].set_xticks(x)
axes[1].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[1].set_title('Class Proportion (%) — Train vs. Test')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Missing Data Summary

In [ ]:
def missing_summary(df, name):
    ms = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
        'Dtype': df.dtypes
    })
    print(f"\n{'='*40}\nMissing data — {name}\n{'='*40}")
    print(ms.to_string())
    return ms

train_missing = missing_summary(train_df[['label', 'description', 'text']], 'Train')
test_missing  = missing_summary(test_df[['label', 'description', 'text']],  'Test')

# Heatmap of missing flags
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, df, title in [
    (axes[0], train_df[['label', 'description', 'text']], 'Train'),
    (axes[1], test_df[['label', 'description', 'text']],  'Test')
]:
    sns.heatmap(df.isnull(), ax=ax, yticklabels=False,
                cbar=False, cmap='OrRd')
    ax.set_title(f'Missing Values Heatmap — {title}')

plt.tight_layout()
plt.show()